In [ ]:
from ScraperFC import FBref
import pandas as pd
import pickle
import ScraperFC as sfc
fbref = FBref()


In [ ]:
#SCRAPER FOR ALL SEASONS

import pickle
import time 
import os

#select years for which you want data
years = ['2012-2013']

for year in years:
    print(f"Starting {year}...")
    all_links = fbref.get_match_links(year=year, league='England Premier League')
    
    new_raw = []
    for i, link in enumerate(all_links):
        try:
            match_data = fbref.scrape_match(link)
            new_raw.append(match_data)
            
            
            time.sleep(3.1) 
            
            if (i + 1) % 10 == 0:
                print(f"Progress: {i + 1}/{len(all_links)} matches scraped.")
                
        except Exception as e:
            print(f"Error on link {link}: {e}")
            continue

    with open(f'raw_{year}.pkl', 'wb') as f:
        pickle.dump(new_raw, f)
    
    print(f"Saved {len(new_raw)} matches for {year}!")

In [ ]:
#retrieve data from raw for a particular year

def flatten_columns(df):
    if isinstance(df.columns[0], tuple):
        df.columns = ['_'.join([str(c) for c in col if not str(c).startswith('Unnamed')]).strip('_') for col in df.columns]
    # if already flat strings, leave them as is
    return df



import re

def flatten_columns(df):
    if isinstance(df.columns[0], tuple):
        df.columns = ['_'.join([str(c) for c in col if not str(c).startswith('Unnamed')]).strip('_') for col in df.columns]
    return df

def convert_match(match):
    rows = []
    
    for team, opp, goals, opp_goals, player_stats in [
        (match.home_team, match.away_team, match.home_goals, match.away_goals, match.home_player_stats),
        (match.away_team, match.home_team, match.away_goals, match.home_goals, match.away_player_stats)
    ]:
        mw_match = re.search(r'Matchweek (\d+)', match.stage)
        matchweek = int(mw_match.group(1)) if mw_match else None
        
        summary = flatten_columns(player_stats['summary'].copy())
        keeper  = flatten_columns(player_stats['keeper'].copy())

        # filter out totals row
        player_col = 'Player' if 'Player' in summary.columns else [c for c in summary.columns if 'layer' in c][0]
        summary = summary[~summary[player_col].astype(str).str.match(r'^\d+ Players$')]

        gf   = int(goals) if goals else 0
        ga   = int(opp_goals) if opp_goals else 0
        sh   = summary.get('Performance_Sh',  summary.get('Sh',   pd.Series([0]))).sum()
        sot  = summary.get('Performance_SoT', summary.get('SoT',  pd.Series([0]))).sum()
        crdy = summary.get('Performance_CrdY',summary.get('CrdY', pd.Series([0]))).sum()
        crdr = summary.get('Performance_CrdR',summary.get('CrdR', pd.Series([0]))).sum()
        xga  = keeper.get('Shot_Stopping_SoTA', keeper.get('SoTA', pd.Series([0]))).sum()

        rows.append({
            'date':          match.date,
            'time':          '15:00',
            'round':         f"Matchweek {matchweek}",
            'day':           pd.Timestamp(match.date).day_name()[:3],
            'venue':         'Home' if team == match.home_team else 'Away',
            'result':        'W' if gf > ga else ('D' if gf == ga else 'L'),
            'gf':            gf,
            'ga':          ga,
            'opponent':      opp,
            'xg':          0,
            'xga':           xga,
            'poss':        0,
            'referee':       match.referee,
            'sh':          sh,
            'sot':           sot,
            'crdy':          crdy,
            'crdr':          crdr,
            '2crdy':         0,
            'season':        2025,
            'team':          team,
        })
    
    return rows

# convert all matches
all_rows = []
for match in raw:
    try:
        all_rows.extend(convert_match(match))
    except Exception as e:
        print(f"Failed: {match.url} — {e}")

df_2526 = pd.DataFrame(all_rows)
print(f"Total rows: {len(df_2526)}")
print(df_2526.head())

In [ ]:
# fix date format
df_2526['date'] = pd.to_datetime(df_2526['date'], format='%A %B %d, %Y').dt.strftime('%d-%m-%Y')

# fix team names
team_name_fix = {
    'Brighton & Hove Albion': 'Brighton and Hove Albion',
}
df_2526['team']     = df_2526['team'].replace(team_name_fix)
df_2526['opponent'] = df_2526['opponent'].replace(team_name_fix)

print(df_2526['date'].head())
print(df_2526['team'].unique())

In [ ]:
#SAVE MATCHES
with open('raw_2025-2026.pkl', 'wb') as f:
    pickle.dump(raw, f)
print(f"Saved {len(raw)} matches!")

In [ ]:
#LOAD MATCHES
import pickle

with open('raw_2025-2026.pkl', 'rb') as f:
    raw = pickle.load(f)

print(f"Loaded {len(raw)} matches!")

In [ ]:
all_links = fbref.get_match_links(year='2025-2026', league='England Premier League')


existing_urls = {match.url for match in raw}

for link in all_links:
    if link not in existing_urls:
        try:
            print(f"Scraping missing match: {link}")
            data = fbref.scrape_match(link)
            raw.append(data)
            
            time.sleep(3.5) 
        except Exception as e:
            print(f"Failed to scrape {link}: {e}")

with open('raw_2025-2026.pkl', 'wb') as f:
    pickle.dump(raw, f)

print(f"Finished! Total matches in raw: {len(raw)}")